In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display
from scipy.stats import pearsonr, spearmanr, mannwhitneyu

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", "{:,.4f}".format)

input_path = Path("/kaggle/input")
csv_files = list(input_path.rglob("*.csv"))

print("CSV files found:")
for i, f in enumerate(csv_files):
    print(i, f)

file_path = csv_files[0]
df = pd.read_csv(file_path, low_memory=False)

print("\nFile path:", file_path)
print("Raw rows:", df.shape[0])
print("Raw columns:", df.shape[1])

display(df.head())

CSV files found:
0 /kaggle/input/datasets/lazurd/fleet-preventative-maintenance/Fleet_Preventative_Maintenance_and_Repair_Work_Orders_20260701.csv

File path: /kaggle/input/datasets/lazurd/fleet-preventative-maintenance/Fleet_Preventative_Maintenance_and_Repair_Work_Orders_20260701.csv
Raw rows: 332420
Raw columns: 37


,UNIQUE_WORK_ORDER_NO,CREATE_DATE,LOC_WORK_ORDER_LOC,LOC_WORK_ORDER_LOC_NAME,WORK_ORDER_YR,WORK_ORDER_NO,JOB_TYPE,EQ_EQUIP_NO,WORK_ORDER_STATUS,METER_1_READING,DATETIME_OUT_SERVICE,DATETIME_IN_SERVICE,DATETIME_OPEN,DATETIME_FIRST_LABOR,DATETIME_FINISHED,DATETIME_CLOSED,DATETIME_UNIT_IN,DATETIME_DUE,DATETIME_PM_SCHED,QTY_EST_HOURS,DOWNTIME_HRS_USER,DOWNTIME_HRS_SHOP,WARRANTY,REAS_REAS_FOR_REPAIR,REAS_FOR_REPAIR_DESC,PRI_PRIORITY_CODE,REF_WORK_ORDER_NO,DEPT_EQUIP_DEPT,DEPT_EQUIP_DEPT_NAME,METER_1_LIFE_TOTAL,EQ_PARENT_EQUIP_NO,DELAY_HOURS,LABOR_HOURS,LABOR_COST,PARTS_COST,COMML_COST,TOTAL_COST
0,2023-3001-1390,2023 Aug 07 12:00:00 AM,3001,WATER WORKS/PROPANE,"2,023","1,390",REPAIR,60959,OPEN,"43,291",2023 Aug 07 12:00:00 AM,NaN,2023 Aug 07 12:00:00 AM,2023 Aug 07 12:00:00 AM,NaN,NaN,2023 Aug 07 12:00:00 AM,2023 Aug 10 12:00:00 AM,NaN,0.0000,0,0,NO,N,NORMAL WEAR (TARGET),3,NaN,0300,WATER WORKS,"43,291",60959,0,2.18,239.8,0,0,239.8
1,2023-4145-273,2023 Aug 07 12:00:00 AM,4145,"NEW EQUIP SHOP - ""J"" ONLY","2,023",273,REPAIR,21951,OPEN,528,2023 Aug 07 12:00:00 AM,NaN,2023 Aug 07 12:00:00 AM,2023 Aug 08 12:00:00 AM,NaN,NaN,2023 Aug 07 12:00:00 AM,2023 Aug 10 12:00:00 AM,NaN,0.0000,0,0,NO,J,PIS-GENERAL (TARGET),3,NaN,0200,PARK DEPARTMENT,528,21951,0,0.5,55,0,0,55
2,2023-2205-312,2023 Aug 07 12:00:00 AM,2205,POLICE DISTRICT 5,"2,023",312,REPAIR,21306,WORK FINISHED,"15,942",2023 Aug 07 12:00:00 AM,2023 Aug 07 12:00:00 AM,2023 Aug 07 12:00:00 AM,2023 Aug 07 12:00:00 AM,2023 Aug 07 12:00:00 AM,NaN,2023 Aug 07 12:00:00 AM,2023 Aug 10 12:00:00 AM,NaN,0.0000,0.05,0.05,YES,N,NORMAL WEAR (TARGET),3,NaN,0222,POLICE,"15,942",21306,0,1,110,0,0,110
3,2023-4131-718,2023 Aug 07 12:00:00 AM,4131,TIRE SHOP,"2,023",718,REPAIR,92830,CLOSED,398,2023 Aug 07 12:00:00 AM,2023 Aug 08 12:00:00 AM,2023 Aug 07 12:00:00 AM,2023 Aug 07 12:00:00 AM,2023 Aug 08 12:00:00 AM,2023 Aug 08 12:00:00 AM,2023 Aug 07 12:00:00 AM,2023 Aug 10 12:00:00 AM,NaN,0.0000,23.18,23.18,NO,N,NORMAL WEAR (TARGET),3,NaN,0253,NEIGHBORHOOD OPERATIONS,398,92830,0,2,220,200.48,4,424.48
4,2023-3001-1389,2023 Aug 07 12:00:00 AM,3001,WATER WORKS/PROPANE,"2,023","1,389",REPAIR,10797,WORK FINISHED,"41,248",2023 Aug 07 12:00:00 AM,2023 Aug 07 12:00:00 AM,2023 Aug 07 12:00:00 AM,2023 Aug 07 12:00:00 AM,2023 Aug 07 12:00:00 AM,NaN,2023 Aug 07 12:00:00 AM,2023 Aug 10 12:00:00 AM,NaN,0.0000,0.03,0.03,NO,N,NORMAL WEAR (TARGET),3,NaN,0300,WATER WORKS,"41,248",10797,0,0.75,82.5,5.97,0,88.47


In [5]:
date_cols = [
    "CREATE_DATE",
    "DATETIME_OUT_SERVICE",
    "DATETIME_IN_SERVICE",
    "DATETIME_OPEN",
    "DATETIME_FIRST_LABOR",
    "DATETIME_FINISHED",
    "DATETIME_CLOSED",
    "DATETIME_UNIT_IN",
    "DATETIME_DUE",
    "DATETIME_PM_SCHED"
]

numeric_cols = [
    "METER_1_READING",
    "QTY_EST_HOURS",
    "DOWNTIME_HRS_USER",
    "DOWNTIME_HRS_SHOP",
    "METER_1_LIFE_TOTAL",
    "DELAY_HOURS",
    "LABOR_HOURS",
    "LABOR_COST",
    "PARTS_COST",
    "COMML_COST",
    "TOTAL_COST"
]

date_format = "%Y %b %d %I:%M:%S %p"

# Convert date fields safely
for col in date_cols:
    if col in df.columns:
        original_text = df[col].astype(str).str.strip()
        
        # First attempt: expected dataset format
        parsed = pd.to_datetime(original_text, format=date_format, errors="coerce")
        
        # Fallback attempt for any values that did not match the expected format
        missing_mask = parsed.isna() & original_text.notna() & (original_text != "nan")
        if missing_mask.any():
            parsed.loc[missing_mask] = pd.to_datetime(
                original_text.loc[missing_mask],
                errors="coerce"
            )
        
        df[col] = parsed

# Convert numeric fields safely
for col in numeric_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.strip()
            .replace({"nan": np.nan, "NaN": np.nan, "None": np.nan, "": np.nan})
        )
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("CREATE_DATE dtype:", df["CREATE_DATE"].dtype)
print("CREATE_DATE start:", df["CREATE_DATE"].min())
print("CREATE_DATE end:", df["CREATE_DATE"].max())
print("Missing CREATE_DATE:", df["CREATE_DATE"].isna().sum())

CREATE_DATE dtype: datetime64[ns]
CREATE_DATE start: 2008-01-01 00:00:00
CREATE_DATE end: 2023-08-09 00:00:00
Missing CREATE_DATE: 0


In [6]:
analysis_start = pd.Timestamp("2018-01-01")
analysis_end_exclusive = pd.Timestamp("2023-01-01")

# Safety check before filtering
if not pd.api.types.is_datetime64_any_dtype(df["CREATE_DATE"]):
    df["CREATE_DATE"] = pd.to_datetime(df["CREATE_DATE"], errors="coerce")

study_df = df[
    (df["CREATE_DATE"] >= analysis_start) &
    (df["CREATE_DATE"] < analysis_end_exclusive)
].copy()

closed_df = study_df[study_df["WORK_ORDER_STATUS"] == "CLOSED"].copy()

closed_df["VALID_LABOR_HOURS"] = closed_df["LABOR_HOURS"].where(closed_df["LABOR_HOURS"] >= 0)
closed_df["VALID_TOTAL_COST"] = closed_df["TOTAL_COST"].where(closed_df["TOTAL_COST"] >= 0)
closed_df["VALID_DOWNTIME_SHOP"] = closed_df["DOWNTIME_HRS_SHOP"].where(closed_df["DOWNTIME_HRS_SHOP"] >= 0)
closed_df["VALID_DOWNTIME_USER"] = closed_df["DOWNTIME_HRS_USER"].where(closed_df["DOWNTIME_HRS_USER"] >= 0)

closed_repair = closed_df[closed_df["JOB_TYPE"] == "REPAIR"].copy()
closed_pm = closed_df[closed_df["JOB_TYPE"] == "PM"].copy()

print("Raw full dataset records:", len(df))
print("Study period records 2018-2022:", len(study_df))
print("Closed records in 2018-2022:", len(closed_df))
print("Closed PM records:", len(closed_pm))
print("Closed REPAIR records:", len(closed_repair))
print("Study period start:", study_df["CREATE_DATE"].min())
print("Study period end:", study_df["CREATE_DATE"].max())

Raw full dataset records: 332420
Study period records 2018-2022: 98557
Closed records in 2018-2022: 98377
Closed PM records: 23334
Closed REPAIR records: 75043
Study period start: 2018-01-01 00:00:00
Study period end: 2022-12-31 00:00:00


In [7]:
raw_audit = pd.DataFrame({
    "Item": [
        "Raw work-order records",
        "Number of columns",
        "Unique work-order numbers",
        "Duplicate work-order numbers",
        "Unique equipment numbers in raw dataset",
        "Raw CREATE_DATE start",
        "Raw CREATE_DATE end"
    ],
    "Result": [
        len(df),
        df.shape[1],
        df["UNIQUE_WORK_ORDER_NO"].nunique(),
        df["UNIQUE_WORK_ORDER_NO"].duplicated().sum(),
        df["EQ_EQUIP_NO"].nunique(),
        df["CREATE_DATE"].min(),
        df["CREATE_DATE"].max()
    ]
})

study_audit = pd.DataFrame({
    "Item": [
        "Study period",
        "Study-period work-order records",
        "Study-period unique work-order numbers",
        "Study-period duplicate work-order numbers",
        "Study-period unique equipment numbers",
        "Study-period CREATE_DATE start",
        "Study-period CREATE_DATE end",
        "Closed work-order records in study period",
        "Closed unique equipment numbers in study period"
    ],
    "Result": [
        "2018-01-01 to 2022-12-31",
        len(study_df),
        study_df["UNIQUE_WORK_ORDER_NO"].nunique(),
        study_df["UNIQUE_WORK_ORDER_NO"].duplicated().sum(),
        study_df["EQ_EQUIP_NO"].nunique(),
        study_df["CREATE_DATE"].min(),
        study_df["CREATE_DATE"].max(),
        len(closed_df),
        closed_df["EQ_EQUIP_NO"].nunique()
    ]
})

display(raw_audit)
display(study_audit)

,Item,Result
0,Raw work-order records,332420
1,Number of columns,37
2,Unique work-order numbers,332420
3,Duplicate work-order numbers,0
4,Unique equipment numbers in raw dataset,8353
5,Raw CREATE_DATE start,2008-01-01 00:00:00
6,Raw CREATE_DATE end,2023-08-09 00:00:00


,Item,Result
0,Study period,2018-01-01 to 2022-12-31
1,Study-period work-order records,98557
2,Study-period unique work-order numbers,98557
3,Study-period duplicate work-order numbers,0
4,Study-period unique equipment numbers,4641
5,Study-period CREATE_DATE start,2018-01-01 00:00:00
6,Study-period CREATE_DATE end,2022-12-31 00:00:00
7,Closed work-order records in study period,98377
8,Closed unique equipment numbers in study period,4630


In [8]:
raw_audit = pd.DataFrame({
    "Item": [
        "Raw work-order records",
        "Number of columns",
        "Unique work-order numbers",
        "Duplicate work-order numbers",
        "Unique equipment numbers in raw dataset",
        "Raw CREATE_DATE start",
        "Raw CREATE_DATE end"
    ],
    "Result": [
        len(df),
        df.shape[1],
        df["UNIQUE_WORK_ORDER_NO"].nunique(),
        df["UNIQUE_WORK_ORDER_NO"].duplicated().sum(),
        df["EQ_EQUIP_NO"].nunique(),
        df["CREATE_DATE"].min(),
        df["CREATE_DATE"].max()
    ]
})

study_audit = pd.DataFrame({
    "Item": [
        "Study period",
        "Study-period work-order records",
        "Study-period unique work-order numbers",
        "Study-period duplicate work-order numbers",
        "Study-period unique equipment numbers",
        "Study-period CREATE_DATE start",
        "Study-period CREATE_DATE end",
        "Closed work-order records in study period",
        "Closed unique equipment numbers in study period"
    ],
    "Result": [
        "2018-01-01 to 2022-12-31",
        len(study_df),
        study_df["UNIQUE_WORK_ORDER_NO"].nunique(),
        study_df["UNIQUE_WORK_ORDER_NO"].duplicated().sum(),
        study_df["EQ_EQUIP_NO"].nunique(),
        study_df["CREATE_DATE"].min(),
        study_df["CREATE_DATE"].max(),
        len(closed_df),
        closed_df["EQ_EQUIP_NO"].nunique()
    ]
})

display(raw_audit)
display(study_audit)

,Item,Result
0,Raw work-order records,332420
1,Number of columns,37
2,Unique work-order numbers,332420
3,Duplicate work-order numbers,0
4,Unique equipment numbers in raw dataset,8353
5,Raw CREATE_DATE start,2008-01-01 00:00:00
6,Raw CREATE_DATE end,2023-08-09 00:00:00


,Item,Result
0,Study period,2018-01-01 to 2022-12-31
1,Study-period work-order records,98557
2,Study-period unique work-order numbers,98557
3,Study-period duplicate work-order numbers,0
4,Study-period unique equipment numbers,4641
5,Study-period CREATE_DATE start,2018-01-01 00:00:00
6,Study-period CREATE_DATE end,2022-12-31 00:00:00
7,Closed work-order records in study period,98377
8,Closed unique equipment numbers in study period,4630


In [9]:
study_job_type_distribution = (
    study_df["JOB_TYPE"]
    .value_counts(dropna=False)
    .reset_index()
)

study_job_type_distribution.columns = ["JOB_TYPE", "Number of records"]
study_job_type_distribution["Percentage"] = (
    study_job_type_distribution["Number of records"] / len(study_df) * 100
).round(2)

study_status_distribution = (
    study_df["WORK_ORDER_STATUS"]
    .value_counts(dropna=False)
    .reset_index()
)

study_status_distribution.columns = ["WORK_ORDER_STATUS", "Number of records"]
study_status_distribution["Percentage"] = (
    study_status_distribution["Number of records"] / len(study_df) * 100
).round(3)

closed_job_type_distribution = (
    closed_df["JOB_TYPE"]
    .value_counts(dropna=False)
    .reset_index()
)

closed_job_type_distribution.columns = ["JOB_TYPE", "Number of records"]
closed_job_type_distribution["Percentage"] = (
    closed_job_type_distribution["Number of records"] / len(closed_df) * 100
).round(2)

display(study_job_type_distribution)
display(study_status_distribution)
display(closed_job_type_distribution)

,JOB_TYPE,Number of records,Percentage
0,REPAIR,75215,76.3200
1,PM,23342,23.6800


,WORK_ORDER_STATUS,Number of records,Percentage
0,CLOSED,98377,99.8170
1,OPEN,151,0.1530
2,WORK FINISHED,29,0.0290


,JOB_TYPE,Number of records,Percentage
0,REPAIR,75043,76.2800
1,PM,23334,23.7200


In [10]:
availability_rows = []

for col in numeric_cols:
    if col in closed_df.columns:
        valid_count = closed_df[col].notna().sum()
        missing_count = closed_df[col].isna().sum()
        negative_count = (closed_df[col] < 0).sum()
        zero_count = (closed_df[col] == 0).sum()
        positive_count = (closed_df[col] > 0).sum()
        
        availability_rows.append({
            "Field": col,
            "Valid values": valid_count,
            "Missing or invalid values": missing_count,
            "Negative values": negative_count,
            "Zero values": zero_count,
            "Positive values": positive_count,
            "Valid percentage": round(valid_count / len(closed_df) * 100, 2)
        })

numeric_data_availability = pd.DataFrame(availability_rows)
numeric_data_availability

,Field,Valid values,Missing or invalid values,Negative values,Zero values,Positive values,Valid percentage
0,METER_1_READING,98377,0,0,6081,92296,100.0000
1,QTY_EST_HOURS,98377,0,0,75993,22384,100.0000
2,DOWNTIME_HRS_USER,98377,0,0,4533,93844,100.0000
3,DOWNTIME_HRS_SHOP,98377,0,0,5419,92958,100.0000
4,METER_1_LIFE_TOTAL,98377,0,0,4533,93844,100.0000
5,DELAY_HOURS,98377,0,0,90994,7383,100.0000
6,LABOR_HOURS,98377,0,1,2011,96365,100.0000
7,LABOR_COST,98377,0,1,3016,95360,100.0000
8,PARTS_COST,98377,0,2,25869,72506,100.0000
9,COMML_COST,98377,0,804,86333,11240,100.0000


In [11]:
closed_date_quality = []

for col in date_cols:
    if col in closed_df.columns:
        closed_date_quality.append({
            "Field": col,
            "Valid dates": closed_df[col].notna().sum(),
            "Missing or invalid dates": closed_df[col].isna().sum(),
            "Start date": closed_df[col].min(),
            "End date": closed_df[col].max()
        })

closed_date_quality = pd.DataFrame(closed_date_quality)
closed_date_quality

,Field,Valid dates,Missing or invalid dates,Start date,End date
0,CREATE_DATE,98377,0,2018-01-01,2022-12-31
1,DATETIME_OUT_SERVICE,98377,0,2011-06-01,2022-12-31
2,DATETIME_IN_SERVICE,98344,33,2017-12-31,2023-08-08
3,DATETIME_OPEN,98377,0,2018-01-01,2022-12-31
4,DATETIME_FIRST_LABOR,96455,1922,2017-12-29,2023-07-20
5,DATETIME_FINISHED,98377,0,2017-12-31,2023-08-08
6,DATETIME_CLOSED,98377,0,2018-01-02,2023-08-08
7,DATETIME_UNIT_IN,98377,0,2011-06-01,2022-12-31
8,DATETIME_DUE,98377,0,2011-06-04,2023-01-03
9,DATETIME_PM_SCHED,23333,75044,2002-03-07,2023-10-28


In [12]:
closed_date_quality = []

for col in date_cols:
    if col in closed_df.columns:
        closed_date_quality.append({
            "Field": col,
            "Valid dates": closed_df[col].notna().sum(),
            "Missing or invalid dates": closed_df[col].isna().sum(),
            "Start date": closed_df[col].min(),
            "End date": closed_df[col].max()
        })

closed_date_quality = pd.DataFrame(closed_date_quality)
closed_date_quality

,Field,Valid dates,Missing or invalid dates,Start date,End date
0,CREATE_DATE,98377,0,2018-01-01,2022-12-31
1,DATETIME_OUT_SERVICE,98377,0,2011-06-01,2022-12-31
2,DATETIME_IN_SERVICE,98344,33,2017-12-31,2023-08-08
3,DATETIME_OPEN,98377,0,2018-01-01,2022-12-31
4,DATETIME_FIRST_LABOR,96455,1922,2017-12-29,2023-07-20
5,DATETIME_FINISHED,98377,0,2017-12-31,2023-08-08
6,DATETIME_CLOSED,98377,0,2018-01-02,2023-08-08
7,DATETIME_UNIT_IN,98377,0,2011-06-01,2022-12-31
8,DATETIME_DUE,98377,0,2011-06-04,2023-01-03
9,DATETIME_PM_SCHED,23333,75044,2002-03-07,2023-10-28


In [13]:
kpi_readiness = pd.DataFrame([
    {
        "Field": "DOWNTIME_HRS_SHOP",
        "KPI use": "Main downtime measure and availability proxy",
        "Usable records": (closed_df["DOWNTIME_HRS_SHOP"].notna() & (closed_df["DOWNTIME_HRS_SHOP"] >= 0)).sum(),
        "Excluded records": (closed_df["DOWNTIME_HRS_SHOP"].isna() | (closed_df["DOWNTIME_HRS_SHOP"] < 0)).sum()
    },
    {
        "Field": "DOWNTIME_HRS_USER",
        "KPI use": "Supporting downtime field",
        "Usable records": (closed_df["DOWNTIME_HRS_USER"].notna() & (closed_df["DOWNTIME_HRS_USER"] >= 0)).sum(),
        "Excluded records": (closed_df["DOWNTIME_HRS_USER"].isna() | (closed_df["DOWNTIME_HRS_USER"] < 0)).sum()
    },
    {
        "Field": "DELAY_HOURS",
        "KPI use": "PM compliance and PM delay",
        "Usable records": (closed_df["DELAY_HOURS"].notna() & (closed_df["DELAY_HOURS"] >= 0)).sum(),
        "Excluded records": (closed_df["DELAY_HOURS"].isna() | (closed_df["DELAY_HOURS"] < 0)).sum()
    },
    {
        "Field": "LABOR_HOURS",
        "KPI use": "Labor-hour analysis",
        "Usable records": (closed_df["LABOR_HOURS"].notna() & (closed_df["LABOR_HOURS"] >= 0)).sum(),
        "Excluded records": (closed_df["LABOR_HOURS"].isna() | (closed_df["LABOR_HOURS"] < 0)).sum()
    },
    {
        "Field": "TOTAL_COST",
        "KPI use": "Maintenance cost analysis",
        "Usable records": (closed_df["TOTAL_COST"].notna() & (closed_df["TOTAL_COST"] >= 0)).sum(),
        "Excluded records": (closed_df["TOTAL_COST"].isna() | (closed_df["TOTAL_COST"] < 0)).sum()
    },
    {
        "Field": "METER_1_READING",
        "KPI use": "Meter-based reliability audit only",
        "Usable records": (closed_df["METER_1_READING"].notna() & (closed_df["METER_1_READING"] > 0)).sum(),
        "Excluded records": (closed_df["METER_1_READING"].isna() | (closed_df["METER_1_READING"] <= 0)).sum()
    },
    {
        "Field": "METER_1_LIFE_TOTAL",
        "KPI use": "Meter-based reliability audit only",
        "Usable records": (closed_df["METER_1_LIFE_TOTAL"].notna() & (closed_df["METER_1_LIFE_TOTAL"] > 0)).sum(),
        "Excluded records": (closed_df["METER_1_LIFE_TOTAL"].isna() | (closed_df["METER_1_LIFE_TOTAL"] <= 0)).sum()
    }
])

kpi_readiness["Usable percentage"] = (
    kpi_readiness["Usable records"] / len(closed_df) * 100
).round(2)

kpi_readiness

,Field,KPI use,Usable records,Excluded records,Usable percentage
0,DOWNTIME_HRS_SHOP,Main downtime measure and availability proxy,98377,0,100.0000
1,DOWNTIME_HRS_USER,Supporting downtime field,98377,0,100.0000
2,DELAY_HOURS,PM compliance and PM delay,98377,0,100.0000
3,LABOR_HOURS,Labor-hour analysis,98376,1,100.0000
4,TOTAL_COST,Maintenance cost analysis,98374,3,100.0000
5,METER_1_READING,Meter-based reliability audit only,92296,6081,93.8200
6,METER_1_LIFE_TOTAL,Meter-based reliability audit only,93844,4533,95.3900


In [14]:
valid_pm_delay = closed_pm[
    closed_pm["DELAY_HOURS"].notna() & 
    (closed_pm["DELAY_HOURS"] >= 0)
].copy()

delayed_pm = valid_pm_delay[valid_pm_delay["DELAY_HOURS"] > 0].copy()

pm_total = len(closed_pm)
pm_valid_delay = len(valid_pm_delay)
pm_missing_delay = closed_pm["DELAY_HOURS"].isna().sum()
pm_on_time = (valid_pm_delay["DELAY_HOURS"] == 0).sum()
pm_delayed = (valid_pm_delay["DELAY_HOURS"] > 0).sum()
pm_negative_delay = (closed_pm["DELAY_HOURS"] < 0).sum()

pm_compliance = round(pm_on_time / pm_valid_delay * 100, 2)

pm_compliance_table = pd.DataFrame({
    "Item": [
        "Closed PM records",
        "PM records with valid DELAY_HOURS",
        "PM records completed without delay",
        "PM records delayed",
        "PM records with missing DELAY_HOURS",
        "PM records with negative DELAY_HOURS",
        "PM compliance (%)"
    ],
    "Result": [
        pm_total,
        pm_valid_delay,
        pm_on_time,
        pm_delayed,
        pm_missing_delay,
        pm_negative_delay,
        pm_compliance
    ]
})

pm_delay_all_valid = valid_pm_delay["DELAY_HOURS"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
).to_frame(name="PM_DELAY_HOURS_ALL_VALID_PM")

pm_delay_delayed_only = delayed_pm["DELAY_HOURS"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
).to_frame(name="PM_DELAY_HOURS_DELAYED_PM_ONLY")

pm_delay_days_delayed_only = (delayed_pm["DELAY_HOURS"] / 24).describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
).to_frame(name="PM_DELAY_DAYS_DELAYED_PM_ONLY")

display(pm_compliance_table)
display(pm_delay_all_valid)
display(pm_delay_delayed_only)
display(pm_delay_days_delayed_only)

,Item,Result
0,Closed PM records,"23,334.0000"
1,PM records with valid DELAY_HOURS,"23,334.0000"
2,PM records completed without delay,"22,473.0000"
3,PM records delayed,861.0000
4,PM records with missing DELAY_HOURS,0.0000
5,PM records with negative DELAY_HOURS,0.0000
6,PM compliance (%),96.3100


,PM_DELAY_HOURS_ALL_VALID_PM
count,"23,334.0000"
mean,6.9049
std,100.2958
min,0.0000
25%,0.0000
50%,0.0000
75%,0.0000
90%,0.0000
95%,0.0000
99%,145.0701


,PM_DELAY_HOURS_DELAYED_PM_ONLY
count,861.0000
mean,187.1300
std,489.0365
min,0.0200
25%,22.4000
50%,67.5200
75%,163.1700
90%,401.8200
95%,764.5300
99%,"2,084.9660"


,PM_DELAY_DAYS_DELAYED_PM_ONLY
count,861.0000
mean,7.7971
std,20.3765
min,0.0008
25%,0.9333
50%,2.8133
75%,6.7987
90%,16.7425
95%,31.8554
99%,86.8736


In [15]:
repair_downtime = closed_repair["DOWNTIME_HRS_SHOP"].where(
    closed_repair["DOWNTIME_HRS_SHOP"] >= 0
)

valid_labor = closed_df["LABOR_HOURS"].where(closed_df["LABOR_HOURS"] >= 0)
valid_cost = closed_df["TOTAL_COST"].where(closed_df["TOTAL_COST"] >= 0)

summary_metrics = pd.DataFrame({
    "Metric": [
        "Total shop downtime hours - all closed records",
        "Total shop downtime hours - REPAIR records",
        "Average shop downtime per REPAIR work order",
        "Total valid labor hours - all closed records",
        "Total valid maintenance cost - all closed records",
        "Average valid total cost per closed work order"
    ],
    "Result": [
        closed_df["VALID_DOWNTIME_SHOP"].sum(),
        repair_downtime.sum(),
        repair_downtime.sum() / len(closed_repair),
        valid_labor.sum(),
        valid_cost.sum(),
        valid_cost.mean()
    ]
})

summary_metrics["Result"] = summary_metrics["Result"].round(2)

summary_metrics_formatted = summary_metrics.copy()
summary_metrics_formatted["Result"] = summary_metrics_formatted["Result"].apply(
    lambda x: f"{x:,.2f}"
)

display(summary_metrics)
display(summary_metrics_formatted)

,Metric,Result
0,Total shop downtime hours - all closed records,"17,406,617.5800"
1,Total shop downtime hours - REPAIR records,"15,366,573.6100"
2,Average shop downtime per REPAIR work order,204.7700
3,Total valid labor hours - all closed records,"356,335.2800"
4,Total valid maintenance cost - all closed records,"67,694,801.8800"
5,Average valid total cost per closed work order,688.1400


,Metric,Result
0,Total shop downtime hours - all closed records,"17,406,617.58"
1,Total shop downtime hours - REPAIR records,"15,366,573.61"
2,Average shop downtime per REPAIR work order,204.77
3,Total valid labor hours - all closed records,"356,335.28"
4,Total valid maintenance cost - all closed records,"67,694,801.88"
5,Average valid total cost per closed work order,688.14


In [16]:
downtime_distribution = repair_downtime.describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
).to_frame(name="REPAIR_DOWNTIME_HRS_SHOP")

labor_distribution = valid_labor.describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
).to_frame(name="VALID_LABOR_HOURS")

cost_distribution = valid_cost.describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
).to_frame(name="VALID_TOTAL_COST")

extreme_values = pd.DataFrame([
    {
        "Field": "REPAIR_DOWNTIME_HRS_SHOP",
        "Condition": "> 720 hours (more than 30 days)",
        "Count": (repair_downtime > 720).sum(),
        "Percentage": round((repair_downtime > 720).sum() / len(closed_repair) * 100, 2)
    },
    {
        "Field": "REPAIR_DOWNTIME_HRS_SHOP",
        "Condition": "> 2160 hours (more than 90 days)",
        "Count": (repair_downtime > 2160).sum(),
        "Percentage": round((repair_downtime > 2160).sum() / len(closed_repair) * 100, 2)
    },
    {
        "Field": "VALID_TOTAL_COST",
        "Condition": "> 10,000",
        "Count": (valid_cost > 10000).sum(),
        "Percentage": round((valid_cost > 10000).sum() / len(closed_df) * 100, 2)
    },
    {
        "Field": "PM_DELAY_HOURS",
        "Condition": "> 720 hours (more than 30 days)",
        "Count": (valid_pm_delay["DELAY_HOURS"] > 720).sum(),
        "Percentage": round((valid_pm_delay["DELAY_HOURS"] > 720).sum() / len(valid_pm_delay) * 100, 2)
    }
])

display(downtime_distribution)
display(labor_distribution)
display(cost_distribution)
display(extreme_values)

,REPAIR_DOWNTIME_HRS_SHOP
count,"75,043.0000"
mean,204.7702
std,664.7934
min,0.0000
25%,0.2500
50%,25.3300
75%,146.5250
90%,495.3880
95%,885.9980
99%,"2,926.0254"


,VALID_LABOR_HOURS
count,"98,376.0000"
mean,3.6222
std,9.7549
min,0.0000
25%,0.8100
50%,1.5000
75%,3.3000
90%,7.1000
95%,11.7500
99%,33.0000


,VALID_TOTAL_COST
count,"98,374.0000"
mean,688.1371
std,"2,036.3356"
min,0.0000
25%,108.0000
50%,231.3000
75%,560.7275
90%,"1,388.0440"
95%,"2,466.3585"
99%,"7,630.1032"


,Field,Condition,Count,Percentage
0,REPAIR_DOWNTIME_HRS_SHOP,> 720 hours (more than 30 days),4894,6.5200
1,REPAIR_DOWNTIME_HRS_SHOP,> 2160 hours (more than 90 days),1060,1.4100
2,VALID_TOTAL_COST,"> 10,000",631,0.6400
3,PM_DELAY_HOURS,> 720 hours (more than 30 days),46,0.2000


In [17]:
def meter_sequence_audit(meter_col):
    meter_repair = closed_df[
        (closed_df["JOB_TYPE"] == "REPAIR") &
        (closed_df["EQ_EQUIP_NO"].notna()) &
        (closed_df["DATETIME_OPEN"].notna())
    ].copy()

    meter_repair = meter_repair.sort_values(["EQ_EQUIP_NO", "DATETIME_OPEN"])

    diff_col = f"{meter_col}_diff"

    meter_repair[diff_col] = (
        meter_repair
        .groupby("EQ_EQUIP_NO")[meter_col]
        .diff()
    )

    meter_repair["calendar_days_between_repairs"] = (
        meter_repair
        .groupby("EQ_EQUIP_NO")["DATETIME_OPEN"]
        .diff()
        .dt.total_seconds() / (24 * 3600)
    )

    positive_diff = meter_repair.loc[meter_repair[diff_col] > 0, diff_col]

    audit = pd.DataFrame({
        "Item": [
            "REPAIR records used for meter sequencing",
            "Records with positive meter difference",
            "Records with zero meter difference",
            "Records with negative meter difference",
            "Records with missing meter difference",
            "Median positive meter difference",
            "Mean positive meter difference",
            "Maximum positive meter difference"
        ],
        "Result": [
            len(meter_repair),
            (meter_repair[diff_col] > 0).sum(),
            (meter_repair[diff_col] == 0).sum(),
            (meter_repair[diff_col] < 0).sum(),
            meter_repair[diff_col].isna().sum(),
            positive_diff.median(),
            positive_diff.mean(),
            positive_diff.max()
        ]
    })

    diff_summary = meter_repair[diff_col].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
    ).to_frame(name=diff_col)

    positive_diff_summary = positive_diff.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
    ).to_frame(name=f"POSITIVE_{diff_col}")

    extreme_meter = pd.DataFrame([
        {
            "Meter field": meter_col,
            "Condition": "> 100,000",
            "Count": (positive_diff > 100000).sum(),
            "Percentage of positive meter differences": round((positive_diff > 100000).sum() / len(positive_diff) * 100, 2)
        },
        {
            "Meter field": meter_col,
            "Condition": "> 500,000",
            "Count": (positive_diff > 500000).sum(),
            "Percentage of positive meter differences": round((positive_diff > 500000).sum() / len(positive_diff) * 100, 2)
        }
    ])

    return meter_repair, audit, diff_summary, positive_diff_summary, extreme_meter


meter_life_repair, meter_life_audit, meter_life_diff_summary, positive_meter_life_diff_summary, extreme_meter_life = meter_sequence_audit("METER_1_LIFE_TOTAL")
meter_reading_repair, meter_reading_audit, meter_reading_diff_summary, positive_meter_reading_diff_summary, extreme_meter_reading = meter_sequence_audit("METER_1_READING")

print("METER_1_LIFE_TOTAL audit")
display(meter_life_audit)
display(meter_life_diff_summary)
display(positive_meter_life_diff_summary)
display(extreme_meter_life)

print("METER_1_READING audit")
display(meter_reading_audit)
display(meter_reading_diff_summary)
display(positive_meter_reading_diff_summary)
display(extreme_meter_reading)

METER_1_LIFE_TOTAL audit


,Item,Result
0,REPAIR records used for meter sequencing,"75,043.0000"
1,Records with positive meter difference,"54,208.0000"
2,Records with zero meter difference,"12,722.0000"
3,Records with negative meter difference,"3,632.0000"
4,Records with missing meter difference,"4,481.0000"
5,Median positive meter difference,522.0000
6,Mean positive meter difference,"2,231.5685"
7,Maximum positive meter difference,"1,057,766.0000"


,METER_1_LIFE_TOTAL_diff
count,"70,562.0000"
mean,979.1932
std,"22,548.4338"
min,"-999,981.0000"
25%,2.0000
50%,259.0000
75%,"1,076.0000"
90%,"2,766.0000"
95%,"4,349.8500"
99%,"12,451.6800"


,POSITIVE_METER_1_LIFE_TOTAL_diff
count,"54,208.0000"
mean,"2,231.5685"
std,"19,639.4664"
min,1.0000
25%,144.0000
50%,522.0000
75%,"1,496.0000"
90%,"3,351.0000"
95%,"5,077.0000"
99%,"16,990.8600"


,Meter field,Condition,Count,Percentage of positive meter differences
0,METER_1_LIFE_TOTAL,"> 100,000",163,0.3000
1,METER_1_LIFE_TOTAL,"> 500,000",25,0.0500


METER_1_READING audit


,Item,Result
0,REPAIR records used for meter sequencing,"75,043.0000"
1,Records with positive meter difference,"54,372.0000"
2,Records with zero meter difference,"12,170.0000"
3,Records with negative meter difference,"4,020.0000"
4,Records with missing meter difference,"4,481.0000"
5,Median positive meter difference,522.0000
6,Mean positive meter difference,"2,333.9848"
7,Maximum positive meter difference,"904,578.0000"


,METER_1_READING_diff
count,"70,562.0000"
mean,874.9979
std,"21,432.8709"
min,"-903,349.0000"
25%,2.0000
50%,260.0000
75%,"1,086.0000"
90%,"2,820.9000"
95%,"4,485.0000"
99%,"16,406.9400"


,POSITIVE_METER_1_READING_diff
count,"54,372.0000"
mean,"2,333.9848"
std,"17,172.2988"
min,1.0000
25%,142.0000
50%,522.0000
75%,"1,511.0000"
90%,"3,407.0000"
95%,"5,226.4500"
99%,"26,294.0400"


,Meter field,Condition,Count,Percentage of positive meter differences
0,METER_1_READING,"> 100,000",208,0.3800
1,METER_1_READING,"> 500,000",19,0.0300


In [18]:
analysis_df = closed_df[
    closed_df["EQ_EQUIP_NO"].notna() &
    closed_df["CREATE_DATE"].notna()
].copy()

analysis_df["MONTH"] = analysis_df["CREATE_DATE"].dt.to_period("M").dt.to_timestamp()
analysis_df["YEAR"] = analysis_df["CREATE_DATE"].dt.year

analysis_df["IS_PM"] = (analysis_df["JOB_TYPE"] == "PM").astype(int)
analysis_df["IS_REPAIR"] = (analysis_df["JOB_TYPE"] == "REPAIR").astype(int)

analysis_df["PM_VALID_DELAY_FLAG"] = (
    (analysis_df["JOB_TYPE"] == "PM") &
    (analysis_df["DELAY_HOURS"].notna()) &
    (analysis_df["DELAY_HOURS"] >= 0)
).astype(int)

analysis_df["PM_ON_TIME_FLAG"] = (
    (analysis_df["JOB_TYPE"] == "PM") &
    (analysis_df["DELAY_HOURS"] == 0)
).astype(int)

analysis_df["PM_DELAY_VALUE"] = np.where(
    (analysis_df["JOB_TYPE"] == "PM") &
    (analysis_df["DELAY_HOURS"].notna()) &
    (analysis_df["DELAY_HOURS"] >= 0),
    analysis_df["DELAY_HOURS"],
    np.nan
)

analysis_df["REPAIR_DOWNTIME_SHOP"] = np.where(
    (analysis_df["JOB_TYPE"] == "REPAIR") &
    (analysis_df["DOWNTIME_HRS_SHOP"].notna()) &
    (analysis_df["DOWNTIME_HRS_SHOP"] >= 0),
    analysis_df["DOWNTIME_HRS_SHOP"],
    np.nan
)

equipment_month = (
    analysis_df
    .groupby(["EQ_EQUIP_NO", "MONTH"], as_index=False)
    .agg(
        work_order_count=("UNIQUE_WORK_ORDER_NO", "count"),
        pm_count=("IS_PM", "sum"),
        repair_count=("IS_REPAIR", "sum"),
        pm_valid_delay_count=("PM_VALID_DELAY_FLAG", "sum"),
        pm_on_time_count=("PM_ON_TIME_FLAG", "sum"),
        pm_delay_hours=("PM_DELAY_VALUE", "mean"),
        downtime_hours=("VALID_DOWNTIME_SHOP", "sum"),
        repair_downtime_hours=("REPAIR_DOWNTIME_SHOP", "sum"),
        labor_hours=("VALID_LABOR_HOURS", "sum"),
        total_cost=("VALID_TOTAL_COST", "sum"),
        valid_cost_records=("VALID_TOTAL_COST", "count")
    )
)

equipment_month["pm_compliance_pct"] = np.where(
    equipment_month["pm_valid_delay_count"] > 0,
    equipment_month["pm_on_time_count"] / equipment_month["pm_valid_delay_count"] * 100,
    np.nan
)

equipment_month["avg_repair_downtime_per_repair_wo"] = np.where(
    equipment_month["repair_count"] > 0,
    equipment_month["repair_downtime_hours"] / equipment_month["repair_count"],
    np.nan
)

equipment_month["pm_to_repair_ratio"] = np.where(
    equipment_month["repair_count"] > 0,
    equipment_month["pm_count"] / equipment_month["repair_count"],
    np.nan
)

equipment_month["calendar_hours"] = equipment_month["MONTH"].dt.days_in_month * 24

equipment_month["availability_proxy_raw_pct"] = (
    (equipment_month["calendar_hours"] - equipment_month["downtime_hours"]) /
    equipment_month["calendar_hours"] * 100
)

equipment_month["availability_proxy_pct"] = equipment_month["availability_proxy_raw_pct"].clip(lower=0, upper=100)

equipment_month["downtime_exceeds_calendar_flag"] = (
    equipment_month["downtime_hours"] > equipment_month["calendar_hours"]
)

equipment_month["repair_downtime_exceeds_calendar_flag"] = (
    equipment_month["repair_downtime_hours"] > equipment_month["calendar_hours"]
)

equipment_month["downtime_hours_capped"] = equipment_month[["downtime_hours", "calendar_hours"]].min(axis=1)

equipment_month["availability_proxy_capped_pct"] = (
    (equipment_month["calendar_hours"] - equipment_month["downtime_hours_capped"]) /
    equipment_month["calendar_hours"] * 100
)

repair_events = analysis_df[
    (analysis_df["JOB_TYPE"] == "REPAIR") &
    (analysis_df["EQ_EQUIP_NO"].notna()) &
    (analysis_df["DATETIME_OPEN"].notna())
].copy()

repair_events = repair_events.sort_values(["EQ_EQUIP_NO", "DATETIME_OPEN"])

repair_events["days_since_previous_repair"] = (
    repair_events
    .groupby("EQ_EQUIP_NO")["DATETIME_OPEN"]
    .diff()
    .dt.total_seconds() / (24 * 3600)
)

repair_events["MONTH"] = repair_events["DATETIME_OPEN"].dt.to_period("M").dt.to_timestamp()

repair_time_between = (
    repair_events
    .groupby(["EQ_EQUIP_NO", "MONTH"], as_index=False)
    .agg(
        calendar_based_time_between_corrective_wos=("days_since_previous_repair", "mean")
    )
)

equipment_month = equipment_month.merge(
    repair_time_between,
    on=["EQ_EQUIP_NO", "MONTH"],
    how="left"
)

print("Equipment-month rows:", len(equipment_month))
display(equipment_month.head())

Equipment-month rows: 55441


,EQ_EQUIP_NO,MONTH,work_order_count,pm_count,repair_count,pm_valid_delay_count,pm_on_time_count,pm_delay_hours,downtime_hours,repair_downtime_hours,labor_hours,total_cost,valid_cost_records,pm_compliance_pct,avg_repair_downtime_per_repair_wo,pm_to_repair_ratio,calendar_hours,availability_proxy_raw_pct,availability_proxy_pct,downtime_exceeds_calendar_flag,repair_downtime_exceeds_calendar_flag,downtime_hours_capped,availability_proxy_capped_pct,calendar_based_time_between_corrective_wos
0,00175,2018-05-01,1,1,0,1,1,0.0000,64.2300,0.0000,1.4900,155.9600,1,100.0000,NaN,NaN,744,91.3669,91.3669,False,False,64.2300,91.3669,NaN
1,00175,2018-07-01,1,0,1,0,0,NaN,0.5500,0.5500,0.2500,22.5000,1,NaN,0.5500,0.0000,744,99.9261,99.9261,False,False,0.5500,99.9261,NaN
2,00175,2018-10-01,3,1,2,1,1,0.0000,191.1300,176.6300,3.0800,377.6100,3,100.0000,88.3150,0.5000,744,74.3105,74.3105,False,False,191.1300,74.3105,54.0000
3,00175,2019-01-01,1,0,1,0,0,NaN,0.0500,0.0500,0.5500,49.5000,1,NaN,0.0500,0.0000,744,99.9933,99.9933,False,False,0.0500,99.9933,74.0000
4,00175,2019-02-01,1,0,1,0,0,NaN,6.3800,6.3800,0.2500,40.9300,1,NaN,6.3800,0.0000,672,99.0506,99.0506,False,False,6.3800,99.0506,25.0000


In [19]:
equipment_month_overview = pd.DataFrame({
    "Item": [
        "Equipment-month observations",
        "Unique equipment units",
        "Start month",
        "End month",
        "Observations with PM records",
        "Observations with REPAIR records",
        "Observations with both PM and REPAIR",
        "Observations with PM compliance value",
        "Observations with availability proxy value"
    ],
    "Result": [
        len(equipment_month),
        equipment_month["EQ_EQUIP_NO"].nunique(),
        equipment_month["MONTH"].min(),
        equipment_month["MONTH"].max(),
        (equipment_month["pm_count"] > 0).sum(),
        (equipment_month["repair_count"] > 0).sum(),
        ((equipment_month["pm_count"] > 0) & (equipment_month["repair_count"] > 0)).sum(),
        equipment_month["pm_compliance_pct"].notna().sum(),
        equipment_month["availability_proxy_capped_pct"].notna().sum()
    ]
})

equipment_month_kpi_summary = equipment_month[
    [
        "pm_count",
        "repair_count",
        "pm_compliance_pct",
        "pm_delay_hours",
        "downtime_hours",
        "avg_repair_downtime_per_repair_wo",
        "labor_hours",
        "total_cost",
        "availability_proxy_capped_pct",
        "pm_to_repair_ratio",
        "calendar_based_time_between_corrective_wos"
    ]
].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).T

display(equipment_month_overview)
display(equipment_month_kpi_summary)

,Item,Result
0,Equipment-month observations,55441
1,Unique equipment units,4630
2,Start month,2018-01-01 00:00:00
3,End month,2022-12-01 00:00:00
4,Observations with PM records,23134
5,Observations with REPAIR records,47602
6,Observations with both PM and REPAIR,15295
7,Observations with PM compliance value,23134
8,Observations with availability proxy value,55441


,count,mean,std,min,25%,50%,75%,90%,95%,99%,max
pm_count,"55,441.0000",0.4209,0.5010,0.0000,0.0000,0.0000,1.0000,1.0000,1.0000,1.0000,3.0000
repair_count,"55,441.0000",1.3536,1.1319,0.0000,1.0000,1.0000,2.0000,3.0000,3.0000,5.0000,22.0000
pm_compliance_pct,"23,134.0000",96.3063,18.8353,0.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000
pm_delay_hours,"23,134.0000",6.8212,99.5799,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,145.0701,"7,657.6700"
downtime_hours,"55,441.0000",313.9665,796.7041,0.0000,21.0300,78.9700,295.4200,761.1100,"1,219.2400","3,982.8200","22,923.9000"
avg_repair_downtime_per_repair_wo,"47,602.0000",257.5773,779.1372,0.0000,6.5700,51.7300,210.4750,575.4300,"1,026.3025","3,837.7481","22,923.9000"
labor_hours,"55,441.0000",6.4273,14.5356,0.0000,1.0000,2.7000,6.7400,14.1000,21.9400,56.5660,536.1200
total_cost,"55,441.0000","1,221.0242","3,013.8650",0.0000,155.6700,413.3200,"1,173.0500","2,744.6300","4,588.7000","12,605.6200","143,227.0100"
availability_proxy_capped_pct,"55,441.0000",72.8031,33.3501,0.0000,59.2513,89.0818,97.1277,99.9819,99.9958,100.0000,100.0000
pm_to_repair_ratio,"47,602.0000",0.2497,0.4026,0.0000,0.0000,0.0000,0.5000,1.0000,1.0000,1.0000,3.0000


In [20]:
equipment_month_valid_downtime = equipment_month[
    equipment_month["downtime_hours"] <= equipment_month["calendar_hours"]
].copy()

robust_availability_summary = pd.DataFrame({
    "Item": [
        "Total equipment-month observations",
        "Observations with downtime <= calendar hours",
        "Observations with downtime > calendar hours",
        "Percentage with downtime > calendar hours",
        "Mean original bounded availability proxy",
        "Mean capped availability proxy",
        "Mean availability proxy after excluding excessive downtime observations"
    ],
    "Result": [
        len(equipment_month),
        len(equipment_month_valid_downtime),
        (equipment_month["downtime_hours"] > equipment_month["calendar_hours"]).sum(),
        round((equipment_month["downtime_hours"] > equipment_month["calendar_hours"]).sum() / len(equipment_month) * 100, 2),
        round(equipment_month["availability_proxy_pct"].mean(), 4),
        round(equipment_month["availability_proxy_capped_pct"].mean(), 4),
        round(equipment_month_valid_downtime["availability_proxy_capped_pct"].mean(), 4)
    ]
})

pm_compliance_variability = pd.DataFrame({
    "Item": [
        "Equipment-month observations",
        "Observations with PM compliance value",
        "Observations with missing PM compliance",
        "PM compliance = 100%",
        "PM compliance = 0%",
        "PM compliance between 0% and 100%",
        "Median PM compliance among valid PM months",
        "Mean PM compliance among valid PM months"
    ],
    "Result": [
        len(equipment_month),
        equipment_month["pm_compliance_pct"].notna().sum(),
        equipment_month["pm_compliance_pct"].isna().sum(),
        (equipment_month["pm_compliance_pct"] == 100).sum(),
        (equipment_month["pm_compliance_pct"] == 0).sum(),
        ((equipment_month["pm_compliance_pct"] > 0) & (equipment_month["pm_compliance_pct"] < 100)).sum(),
        equipment_month["pm_compliance_pct"].median(),
        round(equipment_month["pm_compliance_pct"].mean(), 4)
    ]
})

display(robust_availability_summary)
display(pm_compliance_variability)

,Item,Result
0,Total equipment-month observations,"55,441.0000"
1,Observations with downtime <= calendar hours,"49,634.0000"
2,Observations with downtime > calendar hours,"5,807.0000"
3,Percentage with downtime > calendar hours,10.4700
4,Mean original bounded availability proxy,72.8031
5,Mean capped availability proxy,72.8031
6,Mean availability proxy after excluding excess...,81.3208


,Item,Result
0,Equipment-month observations,"55,441.0000"
1,Observations with PM compliance value,"23,134.0000"
2,Observations with missing PM compliance,"32,307.0000"
3,PM compliance = 100%,"22,275.0000"
4,PM compliance = 0%,850.0000
5,PM compliance between 0% and 100%,9.0000
6,Median PM compliance among valid PM months,100.0000
7,Mean PM compliance among valid PM months,96.3063


In [21]:
yearly_job_type = (
    analysis_df
    .groupby(["YEAR", "JOB_TYPE"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ["PM", "REPAIR"]:
    if col not in yearly_job_type.columns:
        yearly_job_type[col] = 0

yearly_job_type["Total"] = yearly_job_type["PM"] + yearly_job_type["REPAIR"]
yearly_job_type["PM percentage"] = (yearly_job_type["PM"] / yearly_job_type["Total"] * 100).round(2)
yearly_job_type["REPAIR percentage"] = (yearly_job_type["REPAIR"] / yearly_job_type["Total"] * 100).round(2)

yearly_performance = (
    analysis_df
    .groupby("YEAR", as_index=False)
    .agg(
        work_order_count=("UNIQUE_WORK_ORDER_NO", "count"),
        downtime_hours=("VALID_DOWNTIME_SHOP", "sum"),
        labor_hours=("VALID_LABOR_HOURS", "sum"),
        total_cost=("VALID_TOTAL_COST", "sum")
    )
)

monthly_trend = (
    analysis_df
    .groupby(["MONTH", "JOB_TYPE"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ["PM", "REPAIR"]:
    if col not in monthly_trend.columns:
        monthly_trend[col] = 0

monthly_trend["Total"] = monthly_trend["PM"] + monthly_trend["REPAIR"]

monthly_performance = (
    analysis_df
    .groupby("MONTH", as_index=False)
    .agg(
        work_order_count=("UNIQUE_WORK_ORDER_NO", "count"),
        downtime_hours=("VALID_DOWNTIME_SHOP", "sum"),
        labor_hours=("VALID_LABOR_HOURS", "sum"),
        total_cost=("VALID_TOTAL_COST", "sum")
    )
)

display(yearly_job_type)
display(yearly_performance)
display(monthly_trend.head())
display(monthly_performance.head())

JOB_TYPE,YEAR,PM,REPAIR,Total,PM percentage,REPAIR percentage
0,2018,4867,16085,20952,23.2300,76.7700
1,2019,4680,16065,20745,22.5600,77.4400
2,2020,4891,14189,19080,25.6300,74.3700
3,2021,4446,14163,18609,23.8900,76.1100
4,2022,4450,14541,18991,23.4300,76.5700


,YEAR,work_order_count,downtime_hours,labor_hours,total_cost
0,2018,20952,"3,290,498.6600","78,025.1200","13,985,267.1800"
1,2019,20745,"3,666,168.6300","72,775.2400","13,735,828.0900"
2,2020,19080,"3,104,725.0800","69,790.6700","12,819,479.8400"
3,2021,18609,"4,181,343.9000","67,499.5500","12,778,657.6600"
4,2022,18991,"3,163,881.3100","68,244.7000","14,375,569.1100"


JOB_TYPE,MONTH,PM,REPAIR,Total
0,2018-01-01,478,1700,2178
1,2018-02-01,366,1213,1579
2,2018-03-01,442,1324,1766
3,2018-04-01,451,1353,1804
4,2018-05-01,463,1472,1935


,MONTH,work_order_count,downtime_hours,labor_hours,total_cost
0,2018-01-01,2178,"281,297.6000","7,799.9100","1,526,696.8100"
1,2018-02-01,1579,"157,213.7200","5,645.8800","952,594.6000"
2,2018-03-01,1766,"171,606.3900","6,934.5600","1,108,446.5000"
3,2018-04-01,1804,"276,701.1800","6,547.4900","1,115,741.3200"
4,2018-05-01,1935,"436,769.7000","7,443.7300","1,313,335.1300"


In [22]:
equipment_pareto = (
    analysis_df
    .groupby("EQ_EQUIP_NO", as_index=False)
    .agg(
        work_order_count=("UNIQUE_WORK_ORDER_NO", "count"),
        pm_count=("IS_PM", "sum"),
        repair_count=("IS_REPAIR", "sum"),
        downtime_hours=("VALID_DOWNTIME_SHOP", "sum"),
        labor_hours=("VALID_LABOR_HOURS", "sum"),
        total_cost=("VALID_TOTAL_COST", "sum")
    )
)

top_equipment_by_repair = equipment_pareto.sort_values("repair_count", ascending=False).head(20)
top_equipment_by_downtime = equipment_pareto.sort_values("downtime_hours", ascending=False).head(20)
top_equipment_by_cost = equipment_pareto.sort_values("total_cost", ascending=False).head(20)
top_equipment_by_labor = equipment_pareto.sort_values("labor_hours", ascending=False).head(20)

display(top_equipment_by_repair)
display(top_equipment_by_downtime)
display(top_equipment_by_cost)
display(top_equipment_by_labor)

,EQ_EQUIP_NO,work_order_count,pm_count,repair_count,downtime_hours,labor_hours,total_cost
4612,X0252,346,0,346,"5,117.9400","4,853.3600","910,277.1200"
4622,X0400,322,0,322,"8,738.4700","3,097.6000","371,354.0800"
1995,51551,250,19,231,"7,971.9100",856.9000,"169,127.0100"
1629,31548,238,22,216,"5,816.6500","1,052.6900","210,584.4600"
1628,31547,233,20,213,"6,514.8500",889.8400,"170,537.5100"
2510,71549,230,20,210,"11,242.5700",835.8100,"177,852.8900"
1293,21553,224,19,205,"6,645.7500","1,128.6100","260,793.5500"
2509,71548,223,20,203,"8,946.7500","1,012.2900","204,378.8400"
2511,71550,216,20,196,"6,695.4900",897.5300,"182,891.3100"
1626,31545,212,19,193,"8,477.0400",780.0400,"161,035.7200"


,EQ_EQUIP_NO,work_order_count,pm_count,repair_count,downtime_hours,labor_hours,total_cost
4625,X2441,55,0,55,"47,071.7600","1,034.1800","127,577.1000"
325,07580,31,0,31,"40,757.7200",6.0000,"331,426.4700"
4077,S4110,53,0,53,"34,443.3600",273.4000,"101,469.8500"
4620,X0304,123,0,123,"32,222.2200",479.2500,"173,187.4400"
3964,S2201,7,0,7,"31,895.4000",0.2000,"3,820.4700"
3968,S2206,6,0,6,"31,271.6900",0.0000,"5,933.1200"
3967,S2205,9,0,9,"30,767.3100",0.1000,"3,607.7800"
4623,X0442,53,0,53,"30,551.8500","1,781.5000","180,039.8800"
4080,S4130,18,0,18,"29,713.3200",2.7900,"53,098.5300"
4599,X0190,102,0,102,"28,307.9100",726.4800,"90,017.3300"


,EQ_EQUIP_NO,work_order_count,pm_count,repair_count,downtime_hours,labor_hours,total_cost
4612,X0252,346,0,346,"5,117.9400","4,853.3600","910,277.1200"
4616,X0271,171,0,171,"4,163.1800",570.0400,"444,513.8500"
4622,X0400,322,0,322,"8,738.4700","3,097.6000","371,354.0800"
4626,X2444,28,0,28,"19,464.9700","3,896.8600","363,537.6400"
325,07580,31,0,31,"40,757.7200",6.0000,"331,426.4700"
1295,21650,140,6,134,"14,285.4100","1,850.6100","309,435.8600"
509,11671,111,6,105,"17,294.8200","1,155.5600","292,377.3700"
1296,21653,130,6,124,"14,517.8400","1,529.4000","287,460.9600"
3229,91671,143,6,137,"20,914.2300","1,295.8000","282,558.0000"
2516,71652,129,6,123,"19,588.5900","1,636.9300","282,270.4200"


,EQ_EQUIP_NO,work_order_count,pm_count,repair_count,downtime_hours,labor_hours,total_cost
4612,X0252,346,0,346,"5,117.9400","4,853.3600","910,277.1200"
4626,X2444,28,0,28,"19,464.9700","3,896.8600","363,537.6400"
4622,X0400,322,0,322,"8,738.4700","3,097.6000","371,354.0800"
1295,21650,140,6,134,"14,285.4100","1,850.6100","309,435.8600"
4623,X0442,53,0,53,"30,551.8500","1,781.5000","180,039.8800"
2516,71652,129,6,123,"19,588.5900","1,636.9300","282,270.4200"
1296,21653,130,6,124,"14,517.8400","1,529.4000","287,460.9600"
1635,31683,132,6,126,"18,193.5900","1,480.6400","265,965.2600"
157,01651,124,6,118,"15,461.0000","1,374.4500","232,311.6600"
1637,31685,149,6,143,"19,039.1900","1,355.8000","248,021.6600"


In [23]:
department_pareto = None
location_pareto = None

if "DEPT_EQUIP_DEPT_NAME" in analysis_df.columns:
    department_pareto = (
        analysis_df
        .groupby("DEPT_EQUIP_DEPT_NAME", dropna=False, as_index=False)
        .agg(
            work_order_count=("UNIQUE_WORK_ORDER_NO", "count"),
            pm_count=("IS_PM", "sum"),
            repair_count=("IS_REPAIR", "sum"),
            downtime_hours=("VALID_DOWNTIME_SHOP", "sum"),
            labor_hours=("VALID_LABOR_HOURS", "sum"),
            total_cost=("VALID_TOTAL_COST", "sum")
        )
        .sort_values("repair_count", ascending=False)
    )
    display(department_pareto.head(20))

if "LOC_WORK_ORDER_LOC_NAME" in analysis_df.columns:
    location_pareto = (
        analysis_df
        .groupby("LOC_WORK_ORDER_LOC_NAME", dropna=False, as_index=False)
        .agg(
            work_order_count=("UNIQUE_WORK_ORDER_NO", "count"),
            pm_count=("IS_PM", "sum"),
            repair_count=("IS_REPAIR", "sum"),
            downtime_hours=("VALID_DOWNTIME_SHOP", "sum"),
            labor_hours=("VALID_LABOR_HOURS", "sum"),
            total_cost=("VALID_TOTAL_COST", "sum")
        )
        .sort_values("repair_count", ascending=False)
    )
    display(location_pareto.head(20))

,DEPT_EQUIP_DEPT_NAME,work_order_count,pm_count,repair_count,downtime_hours,labor_hours,total_cost
24,POLICE,23809,6021,17788,"3,790,209.5100","60,448.7400","11,180,096.8000"
20,NEIGHBORHOOD OPERATIONS,14623,2016,12607,"1,083,249.9700","55,437.0800","11,844,414.6000"
9,FIRE DEPARTMENT RESPONSE,10972,1377,9595,"1,684,343.0200","79,069.1800","15,407,786.1600"
32,TRAFFIC AND ROAD DIV,10264,1831,8433,"1,511,413.3000","44,916.7300","8,452,968.2200"
19,M.S.D.,12064,3878,8186,"1,927,879.2000","38,099.1400","5,903,101.4000"
36,WATER WORKS,9146,2885,6261,"2,159,918.4500","23,761.8900","4,893,594.5900"
28,RECREATION,4892,1465,3427,"1,107,536.5600","13,606.5200","2,214,304.3400"
22,PARK DEPARTMENT,3222,944,2278,"1,167,711.3000","9,957.6600","2,020,446.1200"
10,FIRE DEPARTMENT SUPPORT,1257,364,893,"305,229.8100","3,058.1400","596,594.1500"
11,FLEET SERVICES,969,192,777,"985,440.3900","4,153.4100","1,289,052.2500"


,LOC_WORK_ORDER_LOC_NAME,work_order_count,pm_count,repair_count,downtime_hours,labor_hours,total_cost
6,LIGHT CAR SHOP,19152,5240,13912,"1,840,879.7000","63,366.3500","10,090,469.8200"
19,SPECIALIZED EQUIPMENT,14294,3363,10931,"3,007,399.9500","55,740.7900","10,332,242.4400"
24,WEST FORK,11323,1520,9803,"351,215.8600","46,434.3100","10,157,446.6900"
21,TIRE SHOP,5518,5,5513,"966,152.4500","8,672.6400","2,978,562.3900"
23,WATER WORKS/PROPANE,7608,2872,4736,"1,436,840.4900","17,139.1900","3,246,121.5800"
4,FIRE SHOP,5010,376,4634,"688,157.9600","56,244.1700","10,674,435.1700"
8,MSD-GALBRAITH,6420,1948,4472,"1,207,394.5500","22,756.8000","3,376,863.4000"
16,POLICE DISTRICT 4,3822,948,2874,"505,345.4600","8,410.5300","1,270,850.9200"
9,MSD-GEST,4798,1937,2861,"520,171.6600","13,492.7700","1,852,908.2300"
1,BODY/PAINT/GLASS SP.,2419,0,2419,"914,670.0800","3,748.4800","2,401,152.9600"


In [24]:
pm_months_only = equipment_month[
    equipment_month["pm_compliance_pct"].notna()
].copy()

pm_months_only["pm_compliance_binary_group"] = np.where(
    pm_months_only["pm_compliance_pct"] == 100,
    "Fully compliant PM month",
    "Delayed or non-compliant PM month"
)

pm_compliance_group_comparison = (
    pm_months_only
    .groupby("pm_compliance_binary_group")
    .agg(
        observations=("EQ_EQUIP_NO", "count"),
        mean_pm_compliance=("pm_compliance_pct", "mean"),
        mean_pm_count=("pm_count", "mean"),
        mean_repair_count=("repair_count", "mean"),
        median_repair_count=("repair_count", "median"),
        mean_downtime_hours=("downtime_hours", "mean"),
        median_downtime_hours=("downtime_hours", "median"),
        mean_avg_repair_downtime=("avg_repair_downtime_per_repair_wo", "mean"),
        median_avg_repair_downtime=("avg_repair_downtime_per_repair_wo", "median"),
        mean_availability_proxy=("availability_proxy_capped_pct", "mean"),
        median_availability_proxy=("availability_proxy_capped_pct", "median"),
        mean_labor_hours=("labor_hours", "mean"),
        mean_total_cost=("total_cost", "mean")
    )
    .reset_index()
)

equipment_month["pm_activity_group"] = np.where(
    equipment_month["pm_count"] > 0,
    "PM activity observed",
    "No PM activity observed"
)

pm_activity_comparison = (
    equipment_month
    .groupby("pm_activity_group")
    .agg(
        observations=("EQ_EQUIP_NO", "count"),
        mean_pm_count=("pm_count", "mean"),
        mean_repair_count=("repair_count", "mean"),
        median_repair_count=("repair_count", "median"),
        mean_downtime_hours=("downtime_hours", "mean"),
        median_downtime_hours=("downtime_hours", "median"),
        mean_avg_repair_downtime=("avg_repair_downtime_per_repair_wo", "mean"),
        median_avg_repair_downtime=("avg_repair_downtime_per_repair_wo", "median"),
        mean_availability_proxy=("availability_proxy_capped_pct", "mean"),
        median_availability_proxy=("availability_proxy_capped_pct", "median"),
        mean_labor_hours=("labor_hours", "mean"),
        mean_total_cost=("total_cost", "mean")
    )
    .reset_index()
)

display(pm_compliance_group_comparison)
display(pm_activity_comparison)

,pm_compliance_binary_group,observations,mean_pm_compliance,mean_pm_count,mean_repair_count,median_repair_count,mean_downtime_hours,median_downtime_hours,mean_avg_repair_downtime,median_avg_repair_downtime,mean_availability_proxy,median_availability_proxy,mean_labor_hours,mean_total_cost
0,Delayed or non-compliant PM month,859,0.5239,1.0128,1.2864,1.0000,356.4011,160.3200,150.7779,47.0300,65.4377,78.3266,6.3139,"1,259.2041"
1,Fully compliant PM month,22275,100.0000,1.0085,1.0868,1.0000,223.6265,73.2500,143.0485,45.2275,76.6638,89.9681,7.5887,"1,293.6313"


,pm_activity_group,observations,mean_pm_count,mean_repair_count,median_repair_count,mean_downtime_hours,median_downtime_hours,mean_avg_repair_downtime,median_avg_repair_downtime,mean_availability_proxy,median_availability_proxy,mean_labor_hours,mean_total_cost
0,No PM activity observed,32307,0.0000,1.5393,1.0000,375.1258,91.0500,311.6656,61.7400,70.3370,87.5726,5.6295,"1,169.9479"
1,PM activity observed,23134,1.0086,1.0942,1.0000,228.5566,74.6700,143.3290,45.3600,76.2469,89.7625,7.5414,"1,292.3530"


In [25]:
def p_format(p):
    if pd.isna(p):
        return "NA"
    if p < 0.001:
        return "<0.001"
    return round(p, 4)

def strength_label(r):
    if pd.isna(r):
        return "Not available"
    abs_r = abs(r)
    if abs_r < 0.10:
        return "Very weak"
    elif abs_r < 0.30:
        return "Weak"
    elif abs_r < 0.50:
        return "Moderate"
    elif abs_r < 0.70:
        return "Strong"
    return "Very strong"

def corr_with_p(data, x, y, label):
    temp = data[[x, y]].replace([np.inf, -np.inf], np.nan).dropna()
    
    if len(temp) < 3:
        return {
            "Relationship": label,
            "X": x,
            "Y": y,
            "N": len(temp),
            "Pearson r": np.nan,
            "Pearson p-value": np.nan,
            "Spearman rho": np.nan,
            "Spearman p-value": np.nan
        }
    
    pearson_r, pearson_p = pearsonr(temp[x], temp[y])
    spearman_rho, spearman_p = spearmanr(temp[x], temp[y])
    
    return {
        "Relationship": label,
        "X": x,
        "Y": y,
        "N": len(temp),
        "Pearson r": pearson_r,
        "Pearson p-value": pearson_p,
        "Spearman rho": spearman_rho,
        "Spearman p-value": spearman_p
    }

In [26]:
hypothesis_tests = [
    ("H1: PM compliance and repair frequency", "pm_compliance_pct", "repair_count"),
    ("H2: PM compliance and capped availability proxy", "pm_compliance_pct", "availability_proxy_capped_pct"),
    ("H3: average repair downtime and capped availability proxy", "avg_repair_downtime_per_repair_wo", "availability_proxy_capped_pct"),
    ("H4: PM-to-repair ratio and downtime", "pm_to_repair_ratio", "downtime_hours"),
    ("Supporting: PM compliance and downtime", "pm_compliance_pct", "downtime_hours"),
    ("Supporting: PM compliance and total cost", "pm_compliance_pct", "total_cost"),
    ("Supporting: PM compliance and labor hours", "pm_compliance_pct", "labor_hours")
]

correlation_with_pvalues = pd.DataFrame([
    corr_with_p(equipment_month_valid_downtime, x, y, label)
    for label, x, y in hypothesis_tests
])

correlation_with_pvalues["Pearson r"] = correlation_with_pvalues["Pearson r"].round(4)
correlation_with_pvalues["Spearman rho"] = correlation_with_pvalues["Spearman rho"].round(4)
correlation_with_pvalues["Pearson p-value"] = correlation_with_pvalues["Pearson p-value"].apply(p_format)
correlation_with_pvalues["Spearman p-value"] = correlation_with_pvalues["Spearman p-value"].apply(p_format)
correlation_with_pvalues["Spearman strength"] = correlation_with_pvalues["Spearman rho"].apply(strength_label)

correlation_with_pvalues

,Relationship,X,Y,N,Pearson r,Pearson p-value,Spearman rho,Spearman p-value,Spearman strength
0,H1: PM compliance and repair frequency,pm_compliance_pct,repair_count,21472,-0.0351,<0.001,-0.0233,<0.001,Very weak
1,H2: PM compliance and capped availability proxy,pm_compliance_pct,availability_proxy_capped_pct,21472,0.0664,<0.001,0.0850,<0.001,Very weak
2,H3: average repair downtime and capped availab...,avg_repair_downtime_per_repair_wo,availability_proxy_capped_pct,42008,-0.8380,<0.001,-0.8834,<0.001,Very strong
3,H4: PM-to-repair ratio and downtime,pm_to_repair_ratio,downtime_hours,42008,0.0360,<0.001,0.1569,<0.001,Weak
4,Supporting: PM compliance and downtime,pm_compliance_pct,downtime_hours,21472,-0.0667,<0.001,-0.0851,<0.001,Very weak
5,Supporting: PM compliance and total cost,pm_compliance_pct,total_cost,21472,-0.0017,0.8065,-0.0229,<0.001,Very weak
6,Supporting: PM compliance and labor hours,pm_compliance_pct,labor_hours,21472,0.0108,0.1138,-0.0054,0.4297,Very weak


In [27]:
pm_base = equipment_month[
    ["EQ_EQUIP_NO", "MONTH", "pm_compliance_pct", "pm_count", "pm_valid_delay_count"]
].copy()

pm_base = pm_base[pm_base["pm_compliance_pct"].notna()].copy()
pm_base["NEXT_MONTH"] = pm_base["MONTH"] + pd.offsets.MonthBegin(1)

next_outcomes = equipment_month[
    [
        "EQ_EQUIP_NO",
        "MONTH",
        "repair_count",
        "downtime_hours",
        "availability_proxy_capped_pct",
        "total_cost",
        "labor_hours",
        "downtime_exceeds_calendar_flag"
    ]
].copy()

next_outcomes = next_outcomes.rename(columns={
    "MONTH": "NEXT_MONTH",
    "repair_count": "next_month_repair_count",
    "downtime_hours": "next_month_downtime_hours",
    "availability_proxy_capped_pct": "next_month_availability_proxy_capped_pct",
    "total_cost": "next_month_total_cost",
    "labor_hours": "next_month_labor_hours",
    "downtime_exceeds_calendar_flag": "next_month_downtime_exceeds_calendar_flag"
})

lag_df = pm_base.merge(
    next_outcomes,
    on=["EQ_EQUIP_NO", "NEXT_MONTH"],
    how="left"
)

lag_df_valid = lag_df[
    (lag_df["next_month_repair_count"].notna()) &
    (lag_df["next_month_downtime_exceeds_calendar_flag"] != True)
].copy()

lag_overview = pd.DataFrame({
    "Item": [
        "PM equipment-month observations",
        "Observations with next-month outcomes",
        "Observations after excluding next-month excessive downtime",
        "Excluded next-month excessive downtime observations"
    ],
    "Result": [
        len(lag_df),
        lag_df["next_month_repair_count"].notna().sum(),
        len(lag_df_valid),
        lag_df["next_month_downtime_exceeds_calendar_flag"].eq(True).sum()
    ]
})

lag_tests = [
    ("Lag: PM compliance and next-month repair count", "pm_compliance_pct", "next_month_repair_count"),
    ("Lag: PM compliance and next-month downtime", "pm_compliance_pct", "next_month_downtime_hours"),
    ("Lag: PM compliance and next-month capped availability", "pm_compliance_pct", "next_month_availability_proxy_capped_pct"),
    ("Lag: PM compliance and next-month total cost", "pm_compliance_pct", "next_month_total_cost"),
    ("Lag: PM compliance and next-month labor hours", "pm_compliance_pct", "next_month_labor_hours")
]

lag_correlation_with_pvalues = pd.DataFrame([
    corr_with_p(lag_df_valid, x, y, label)
    for label, x, y in lag_tests
])

lag_correlation_with_pvalues["Pearson r"] = lag_correlation_with_pvalues["Pearson r"].round(4)
lag_correlation_with_pvalues["Spearman rho"] = lag_correlation_with_pvalues["Spearman rho"].round(4)
lag_correlation_with_pvalues["Pearson p-value"] = lag_correlation_with_pvalues["Pearson p-value"].apply(p_format)
lag_correlation_with_pvalues["Spearman p-value"] = lag_correlation_with_pvalues["Spearman p-value"].apply(p_format)
lag_correlation_with_pvalues["Spearman strength"] = lag_correlation_with_pvalues["Spearman rho"].apply(strength_label)

display(lag_overview)
display(lag_correlation_with_pvalues)

,Item,Result
0,PM equipment-month observations,23134
1,Observations with next-month outcomes,7476
2,Observations after excluding next-month excess...,6783
3,Excluded next-month excessive downtime observa...,693


,Relationship,X,Y,N,Pearson r,Pearson p-value,Spearman rho,Spearman p-value,Spearman strength
0,Lag: PM compliance and next-month repair count,pm_compliance_pct,next_month_repair_count,6783,0.0339,0.0052,0.0294,0.0156,Very weak
1,Lag: PM compliance and next-month downtime,pm_compliance_pct,next_month_downtime_hours,6783,-0.0417,<0.001,-0.0354,0.0036,Very weak
2,Lag: PM compliance and next-month capped avail...,pm_compliance_pct,next_month_availability_proxy_capped_pct,6783,0.0405,<0.001,0.0345,0.0045,Very weak
3,Lag: PM compliance and next-month total cost,pm_compliance_pct,next_month_total_cost,6783,0.0272,0.0252,0.0429,<0.001,Very weak
4,Lag: PM compliance and next-month labor hours,pm_compliance_pct,next_month_labor_hours,6783,0.0366,0.0025,0.0470,<0.001,Very weak


In [28]:
group_test_df = pm_months_only.copy()

outcome_vars = [
    "repair_count",
    "downtime_hours",
    "avg_repair_downtime_per_repair_wo",
    "availability_proxy_capped_pct",
    "labor_hours",
    "total_cost"
]

mann_whitney_results = []

for var in outcome_vars:
    g1 = group_test_df.loc[
        group_test_df["pm_compliance_binary_group"] == "Fully compliant PM month",
        var
    ].replace([np.inf, -np.inf], np.nan).dropna()
    
    g2 = group_test_df.loc[
        group_test_df["pm_compliance_binary_group"] == "Delayed or non-compliant PM month",
        var
    ].replace([np.inf, -np.inf], np.nan).dropna()
    
    if len(g1) > 0 and len(g2) > 0:
        stat, p = mannwhitneyu(g1, g2, alternative="two-sided")
    else:
        stat, p = np.nan, np.nan
    
    mann_whitney_results.append({
        "Outcome": var,
        "Fully compliant N": len(g1),
        "Delayed/non-compliant N": len(g2),
        "Fully compliant median": g1.median(),
        "Delayed/non-compliant median": g2.median(),
        "Fully compliant mean": g1.mean(),
        "Delayed/non-compliant mean": g2.mean(),
        "Mann-Whitney U": stat,
        "p-value": p
    })

mann_whitney_results = pd.DataFrame(mann_whitney_results)

for col in [
    "Fully compliant median",
    "Delayed/non-compliant median",
    "Fully compliant mean",
    "Delayed/non-compliant mean",
    "Mann-Whitney U"
]:
    mann_whitney_results[col] = mann_whitney_results[col].round(4)

mann_whitney_results["p-value"] = mann_whitney_results["p-value"].apply(p_format)

mann_whitney_results

,Outcome,Fully compliant N,Delayed/non-compliant N,Fully compliant median,Delayed/non-compliant median,Fully compliant mean,Delayed/non-compliant mean,Mann-Whitney U,p-value
0,repair_count,22275,859,1.0000,1.0000,1.0868,1.2864,"8,984,622.0000",0.0013
1,downtime_hours,22275,859,73.2500,160.3200,223.6265,356.4011,"6,982,138.0000",<0.001
2,avg_repair_downtime_per_repair_wo,14740,555,45.2275,47.0300,143.0485,150.7779,"3,961,657.5000",0.2076
3,availability_proxy_capped_pct,22275,859,89.9681,78.3266,76.6638,65.4377,"12,150,839.0000",<0.001
4,labor_hours,22275,859,3.5000,3.6600,7.5887,6.3139,"9,564,637.5000",0.9897
5,total_cost,22275,859,467.5800,585.5600,"1,293.6313","1,259.2041","9,003,326.0000",0.0033


In [30]:
output_dir = Path("/kaggle/working/chapter4_outputs_2018_2022")
output_dir.mkdir(parents=True, exist_ok=True)

tables_to_save = {
    "raw_audit": raw_audit,
    "study_audit": study_audit,
    "study_job_type_distribution": study_job_type_distribution,
    "study_status_distribution": study_status_distribution,
    "closed_job_type_distribution": closed_job_type_distribution,
    "equipment_coverage": equipment_coverage,
    "numeric_data_availability": numeric_data_availability,
    "closed_date_quality": closed_date_quality,
    "kpi_readiness": kpi_readiness,
    "pm_compliance_table": pm_compliance_table,
    "pm_delay_all_valid": pm_delay_all_valid,
    "pm_delay_delayed_only": pm_delay_delayed_only,
    "pm_delay_days_delayed_only": pm_delay_days_delayed_only,
    "summary_metrics": summary_metrics,
    "summary_metrics_formatted": summary_metrics_formatted,
    "downtime_distribution": downtime_distribution,
    "labor_distribution": labor_distribution,
    "cost_distribution": cost_distribution,
    "extreme_values": extreme_values,
    "meter_life_audit": meter_life_audit,
    "meter_life_diff_summary": meter_life_diff_summary,
    "positive_meter_life_diff_summary": positive_meter_life_diff_summary,
    "extreme_meter_life": extreme_meter_life,
    "meter_reading_audit": meter_reading_audit,
    "meter_reading_diff_summary": meter_reading_diff_summary,
    "positive_meter_reading_diff_summary": positive_meter_reading_diff_summary,
    "extreme_meter_reading": extreme_meter_reading,
    "equipment_month_overview": equipment_month_overview,
    "equipment_month_kpi_summary": equipment_month_kpi_summary,
    "robust_availability_summary": robust_availability_summary,
    "pm_compliance_variability": pm_compliance_variability,
    "yearly_job_type": yearly_job_type,
    "yearly_performance": yearly_performance,
    "monthly_trend": monthly_trend,
    "monthly_performance": monthly_performance,
    "equipment_pareto": equipment_pareto,
    "top_equipment_by_repair": top_equipment_by_repair,
    "top_equipment_by_downtime": top_equipment_by_downtime,
    "top_equipment_by_cost": top_equipment_by_cost,
    "top_equipment_by_labor": top_equipment_by_labor,
    "pm_compliance_group_comparison": pm_compliance_group_comparison,
    "pm_activity_comparison": pm_activity_comparison,
    "correlation_with_pvalues": correlation_with_pvalues,
    "lag_overview": lag_overview,
    "lag_correlation_with_pvalues": lag_correlation_with_pvalues,
    "mann_whitney_results": mann_whitney_results
}

if department_pareto is not None:
    tables_to_save["department_pareto"] = department_pareto

if location_pareto is not None:
    tables_to_save["location_pareto"] = location_pareto

for name, table in tables_to_save.items():
    table.to_csv(output_dir / f"{name}.csv", index=True)

equipment_month.to_csv(output_dir / "equipment_month_dataset_2018_2022.csv", index=False)
analysis_df.to_csv(output_dir / "closed_cleaned_work_orders_2018_2022.csv", index=False)

print("Saved outputs to:", output_dir)
print("Saved tables:", len(tables_to_save) + 2)

NameError: name 'equipment_coverage' is not defined